In [ ]:
%pip install soundfile
%pip install torch

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ------------------------------ --------- 0.8/1.0 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 8.0 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ============================================================
# PART 1: E-DAIC DATASET INSPECTION
# ============================================================

import os
import glob
import pandas as pd
import soundfile as sf


# ---------- Dataset path ----------

ROOT = r"E:\EDAIC-WOZ\data"


# ---------- 0. GPU check ----------

try:
    import torch

    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print(
            "GPU:",
            torch.cuda.get_device_name(0)
        )

except ImportError:
    print("PyTorch not installed yet (fine for now)")


# ---------- 1. Participant folders ----------

folders = sorted(
    f
    for f in glob.glob(
        os.path.join(ROOT, "*_P")
    )
    if os.path.isdir(f)
)

print(
    f"\nParticipant folders found: {len(folders)}"
)

if not folders:
    raise SystemExit(
        "No '*_P' folders found. Check ROOT."
    )


# ---------- 2. Inspect first participant ----------

first = folders[0]

print("\nFiles inside:", first)

for dirpath, _, files in os.walk(first):

    for f in sorted(files):

        print(
            "  ",
            os.path.relpath(
                os.path.join(dirpath, f),
                first
            )
        )


# ---------- 3. Find transcript ----------

transcript_files = glob.glob(
    os.path.join(
        first,
        "*_Transcript.csv"
    )
)

if transcript_files:

    tpath = transcript_files[0]

    transcript = pd.read_csv(tpath)

    print("\n--- Transcript ---")

    print(
        "Path:",
        tpath
    )

    print(
        "Columns:",
        list(transcript.columns)
    )

    print("\nFirst 10 rows:")

    print(
        transcript
        .head(10)
        .to_string(index=False)
    )

    print(
        "\nRows:",
        len(transcript)
    )

    # Transcript duration
    if "End_Time" in transcript.columns:

        print(
            "Last transcript end time (s):",
            transcript["End_Time"].iloc[-1]
        )

else:

    print(
        "\nNo transcript file found in:",
        first
    )


# ---------- 4. Find audio ----------

audio_files = glob.glob(
    os.path.join(
        first,
        "*_AUDIO.wav"
    )
)

if audio_files:

    wpath = audio_files[0]

    print("\n--- Audio ---")

    print(
        "Path:",
        wpath
    )

    # Read only WAV metadata.
    # Does NOT load the entire audio into memory.
    info = sf.info(wpath)

    print(
        f"Duration: {info.duration:.1f} seconds"
    )

    print(
        "Sample rate:",
        info.samplerate
    )

    print(
        "Channels:",
        info.channels
    )

else:

    print(
        "\nNo WAV file found in:",
        first
    )

CUDA available: False

Participant folders found: 275

Files inside: E:\EDAIC-WOZ\data\300_P
   300_AUDIO.wav
   300_Transcript.csv
   features\300_BoAW_openSMILE_2.3.0_MFCC.csv
   features\300_BoAW_openSMILE_2.3.0_eGeMAPS.csv
   features\300_BoVW_openFace_2.1.0_Pose_Gaze_AUs.csv
   features\300_CNN_ResNet.mat
   features\300_CNN_VGG.mat
   features\300_OpenFace2.1.0_Pose_gaze_AUs.csv
   features\300_OpenSMILE2.3.0_egemaps.csv
   features\300_OpenSMILE2.3.0_mfcc.csv
   features\300_densenet201.csv
   features\300_vgg16.csv

--- Transcript ---
Path: E:\EDAIC-WOZ\data\300_P\300_Transcript.csv
Columns: ['Start_Time', 'End_Time', 'Text', 'Confidence']

First 10 rows:
 Start_Time  End_Time                                         Text  Confidence
       14.3      15.1                              so I'm going to    0.934210
       20.3      21.1                         interview in Spanish    0.608470
       23.9      24.3                                         okay    0.690606
       62.1 

In [2]:
# ============================================================
# PART 2: BUILD E-DAIC MANIFEST
# ============================================================

import fnmatch
import glob
import os
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

ROOT = r"E:\EDAIC-WOZ\data"

LABEL_ROOT = r"E:\EDAIC-WOZ\labels"

MANIFEST = os.path.join(
    ROOT,
    "manifest.csv"
)


print("Data folder:")
print(ROOT)

print("\nLabel folder:")
print(LABEL_ROOT)


# ============================================================
# 2. CHECK PATHS
# ============================================================

if not os.path.isdir(ROOT):
    raise SystemExit(
        f"ERROR: Data folder not found:\n{ROOT}"
    )

if not os.path.isdir(LABEL_ROOT):
    raise SystemExit(
        f"ERROR: Label folder not found:\n{LABEL_ROOT}"
    )


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def find_col(df, keyword):
    """
    Find the first column containing keyword.
    """

    for c in df.columns:

        if keyword.lower() in c.lower():
            return c

    return None


def find_one(folder, pattern, exclude=()):
    """
    Find the first file inside folder matching pattern.

    Matching is case-insensitive.
    """

    if folder is None:
        return None

    for dirpath, _, files in os.walk(folder):

        for f in sorted(files):

            name = f.lower()

            if fnmatch.fnmatch(
                name,
                pattern.lower()
            ):

                if not any(
                    x.lower() in name
                    for x in exclude
                ):
                    return os.path.join(
                        dirpath,
                        f
                    )

    return None


# ============================================================
# 4. LOAD LABEL FILE
# ============================================================

def load_labels(split):

    label_file = os.path.join(
        LABEL_ROOT,
        f"{split}_split.csv"
    )

    print("\n" + "-" * 60)
    print(f"Loading {split.upper()} labels")
    print("-" * 60)

    if not os.path.isfile(label_file):

        print(
            f"WARNING: File not found:\n{label_file}"
        )

        return None

    df = pd.read_csv(label_file)

    print("File:", label_file)
    print("Shape:", df.shape)

    print(
        "Columns:",
        df.columns.tolist()
    )

    # Find columns
    id_col = find_col(
        df,
        "participant"
    )

    bin_col = find_col(
        df,
        "binary"
    )

    score_col = find_col(
        df,
        "score"
    )

    if id_col is None:

        raise ValueError(
            f"Participant ID column not found "
            f"in {label_file}\n"
            f"Available columns: {df.columns.tolist()}"
        )

    if bin_col is None:

        raise ValueError(
            f"PHQ binary column not found "
            f"in {label_file}"
        )

    if score_col is None:

        raise ValueError(
            f"PHQ score column not found "
            f"in {label_file}"
        )

    print("\nDetected columns:")

    print("Participant ID:", id_col)
    print("PHQ Binary:", bin_col)
    print("PHQ Score:", score_col)

    # Create simplified dataframe
    result = pd.DataFrame({

        "pid":
            pd.to_numeric(
                df[id_col],
                errors="coerce"
            ),

        "split":
            split,

        "label":
            pd.to_numeric(
                df[bin_col],
                errors="coerce"
            ),

        "phq":
            pd.to_numeric(
                df[score_col],
                errors="coerce"
            )
    })

    # Remove invalid participant IDs
    result = result.dropna(
        subset=["pid"]
    )

    result["pid"] = result[
        "pid"
    ].astype(int)

    return result


# ============================================================
# 5. LOAD TRAIN / DEV / TEST
# ============================================================

frames = []

for split in [
    "train",
    "dev",
    "test"
]:

    result = load_labels(split)

    if result is not None:
        frames.append(result)


if not frames:

    raise SystemExit(
        "\nERROR: No label files were loaded."
    )


labels = pd.concat(
    frames,
    ignore_index=True
)


# ============================================================
# 6. LABEL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("LABEL SUMMARY")
print("=" * 60)

print(
    "\nTotal labeled participants:",
    len(labels)
)

print("\nParticipants per split:")

print(
    labels["split"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nPHQ Binary distribution:")

print(
    labels["label"]
    .value_counts(
        dropna=False
    )
    .sort_index()
    .to_string()
)

print("\nPHQ Score statistics:")

print(
    labels["phq"]
    .describe()
    .to_string()
)


# ============================================================
# 7. FIND PARTICIPANT FOLDERS
# ============================================================

print("\n" + "=" * 60)
print("PARTICIPANT FOLDERS")
print("=" * 60)

folders = sorted(
    f
    for f in glob.glob(
        os.path.join(
            ROOT,
            "*_P"
        )
    )
    if os.path.isdir(f)
)

print(
    "Participant folders found:",
    len(folders)
)


if not folders:

    raise SystemExit(
        "ERROR: No participant folders found."
    )


# ============================================================
# 8. MAP PARTICIPANT ID → FOLDER
# ============================================================

folder_by_pid = {}

for folder in folders:

    folder_name = os.path.basename(
        folder
    )

    try:

        # Example:
        # 300_P → 300

        pid = int(
            folder_name.split("_")[0]
        )

        folder_by_pid[pid] = folder

    except ValueError:

        print(
            "WARNING: Could not extract ID from:",
            folder_name
        )


print(
    "Participant IDs successfully mapped:",
    len(folder_by_pid)
)


# ============================================================
# 9. BUILD MANIFEST
# ============================================================

print("\n" + "=" * 60)
print("BUILDING MANIFEST")
print("=" * 60)

rows = []

for r in labels.itertuples(
    index=False
):

    folder = folder_by_pid.get(
        r.pid
    )

    # Audio
    wav_path = find_one(
        folder,
        "*.wav"
    )

    # Transcript
    transcript_path = find_one(
        folder,
        "*transcript*.csv"
    )

    # Precomputed eGeMAPS
    egemaps_path = find_one(
        folder,
        "*egemaps*.csv",
        exclude=("boaw",)
    )

    # Precomputed MFCC
    mfcc_path = find_one(
        folder,
        "*mfcc*.csv",
        exclude=("boaw",)
    )

    rows.append({

        "pid":
            r.pid,

        "split":
            r.split,

        "label":
            r.label,

        "phq":
            r.phq,

        "wav":
            wav_path,

        "transcript":
            transcript_path,

        "egemaps":
            egemaps_path,

        "mfcc":
            mfcc_path
    })


# ============================================================
# 10. CREATE MANIFEST DATAFRAME
# ============================================================

m = pd.DataFrame(rows)


# ============================================================
# 11. CHECK REQUIRED FILES
# ============================================================

m["has_audio"] = (
    m["wav"].notna()
)

m["has_transcript"] = (
    m["transcript"].notna()
)

m["ok"] = (
    m["has_audio"]
    &
    m["has_transcript"]
)


# ============================================================
# 12. MANIFEST SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("MANIFEST SUMMARY")
print("=" * 60)

summary = (
    m.groupby("split")
    .agg(
        participants=("pid", "size"),
        audio=("has_audio", "sum"),
        transcript=("has_transcript", "sum"),
        ok=("ok", "sum"),
        depressed=("label", "sum")
    )
)

print(
    summary.to_string()
)


# ============================================================
# 13. CHECK FOR MISSING DATA
# ============================================================

missing = m[
    ~m["ok"]
]

if len(missing) > 0:

    print("\n" + "=" * 60)
    print("MISSING AUDIO / TRANSCRIPTS")
    print("=" * 60)

    print(
        f"\n{len(missing)} participants "
        "are missing required files:\n"
    )

    print(
        missing[
            [
                "pid",
                "split",
                "wav",
                "transcript"
            ]
        ].to_string(index=False)
    )

else:

    print(
        "\nAll labeled participants have "
        "both audio and transcript."
    )


# ============================================================
# 14. CHECK FOR LABELLED PARTICIPANTS WITHOUT FOLDERS
# ============================================================

missing_folders = labels[
    ~labels["pid"].isin(
        folder_by_pid.keys()
    )
]

if len(missing_folders) > 0:

    print("\n" + "=" * 60)
    print("MISSING PARTICIPANT FOLDERS")
    print("=" * 60)

    print(
        f"\n{len(missing_folders)} labeled "
        "participants have no matching folder:\n"
    )

    print(
        missing_folders[
            [
                "pid",
                "split",
                "label",
                "phq"
            ]
        ].to_string(index=False)
    )

else:

    print(
        "\nEvery labeled participant has "
        "a matching participant folder."
    )


# ============================================================
# 15. SAVE MANIFEST
# ============================================================

m.to_csv(
    MANIFEST,
    index=False
)

print("\n" + "=" * 60)
print("MANIFEST SAVED")
print("=" * 60)

print(
    "\nPath:",
    MANIFEST
)

print(
    "Rows:",
    len(m)
)


# ============================================================
# 16. PREVIEW
# ============================================================

print("\n" + "=" * 60)
print("MANIFEST PREVIEW")
print("=" * 60)

display(
    m.head(10)
)

Data folder:
E:\EDAIC-WOZ\data

Label folder:
E:\EDAIC-WOZ\labels

------------------------------------------------------------
Loading TRAIN labels
------------------------------------------------------------
File: E:\EDAIC-WOZ\labels\train_split.csv
Shape: (163, 6)
Columns: ['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']

Detected columns:
Participant ID: Participant_ID
PHQ Binary: PHQ_Binary
PHQ Score: PHQ_Score

------------------------------------------------------------
Loading DEV labels
------------------------------------------------------------
File: E:\EDAIC-WOZ\labels\dev_split.csv
Shape: (56, 6)
Columns: ['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']

Detected columns:
Participant ID: Participant_ID
PHQ Binary: PHQ_Binary
PHQ Score: PHQ_Score

------------------------------------------------------------
Loading TEST labels
------------------------------------------------------------
File: E:

,pid,split,label,phq,wav,transcript,egemaps,mfcc,has_audio,has_transcript,ok
0,302,train,0,4,E:\EDAIC-WOZ\data\302_P\302_AUDIO.wav,E:\EDAIC-WOZ\data\302_P\302_Transcript.csv,E:\EDAIC-WOZ\data\302_P\features\302_OpenSMILE...,E:\EDAIC-WOZ\data\302_P\features\302_OpenSMILE...,True,True,True
1,303,train,0,0,E:\EDAIC-WOZ\data\303_P\303_AUDIO.wav,E:\EDAIC-WOZ\data\303_P\303_Transcript.csv,E:\EDAIC-WOZ\data\303_P\features\303_OpenSMILE...,E:\EDAIC-WOZ\data\303_P\features\303_OpenSMILE...,True,True,True
2,304,train,0,6,E:\EDAIC-WOZ\data\304_P\304_AUDIO.wav,E:\EDAIC-WOZ\data\304_P\304_Transcript.csv,E:\EDAIC-WOZ\data\304_P\features\304_OpenSMILE...,E:\EDAIC-WOZ\data\304_P\features\304_OpenSMILE...,True,True,True
3,305,train,0,7,E:\EDAIC-WOZ\data\305_P\305_AUDIO.wav,E:\EDAIC-WOZ\data\305_P\305_Transcript.csv,E:\EDAIC-WOZ\data\305_P\features\305_OpenSMILE...,E:\EDAIC-WOZ\data\305_P\features\305_OpenSMILE...,True,True,True
4,307,train,0,4,E:\EDAIC-WOZ\data\307_P\307_AUDIO.wav,E:\EDAIC-WOZ\data\307_P\307_Transcript.csv,E:\EDAIC-WOZ\data\307_P\features\307_OpenSMILE...,E:\EDAIC-WOZ\data\307_P\features\307_OpenSMILE...,True,True,True
5,308,train,1,22,E:\EDAIC-WOZ\data\308_P\308_AUDIO.wav,E:\EDAIC-WOZ\data\308_P\308_Transcript.csv,E:\EDAIC-WOZ\data\308_P\features\308_OpenSMILE...,E:\EDAIC-WOZ\data\308_P\features\308_OpenSMILE...,True,True,True
6,309,train,1,15,E:\EDAIC-WOZ\data\309_P\309_AUDIO.wav,E:\EDAIC-WOZ\data\309_P\309_Transcript.csv,E:\EDAIC-WOZ\data\309_P\features\309_OpenSMILE...,E:\EDAIC-WOZ\data\309_P\features\309_OpenSMILE...,True,True,True
7,310,train,0,4,E:\EDAIC-WOZ\data\310_P\310_AUDIO.wav,E:\EDAIC-WOZ\data\310_P\310_Transcript.csv,E:\EDAIC-WOZ\data\310_P\features\310_OpenSMILE...,E:\EDAIC-WOZ\data\310_P\features\310_OpenSMILE...,True,True,True
8,311,train,1,21,E:\EDAIC-WOZ\data\311_P\311_AUDIO.wav,E:\EDAIC-WOZ\data\311_P\311_Transcript.csv,E:\EDAIC-WOZ\data\311_P\features\311_OpenSMILE...,E:\EDAIC-WOZ\data\311_P\features\311_OpenSMILE...,True,True,True
9,312,train,0,2,E:\EDAIC-WOZ\data\312_P\312_AUDIO.wav,E:\EDAIC-WOZ\data\312_P\312_Transcript.csv,E:\EDAIC-WOZ\data\312_P\features\312_OpenSMILE...,E:\EDAIC-WOZ\data\312_P\features\312_OpenSMILE...,True,True,True


In [3]:
# ============================================================
# PART 2.5: VERIFY MANIFEST AGAINST ACTUAL DATA
# ============================================================

import os
import pandas as pd

ROOT = r"E:\EDAIC-WOZ\data"
MANIFEST = os.path.join(ROOT, "manifest.csv")

m = pd.read_csv(MANIFEST)

print("=" * 70)
print("VERIFYING MANIFEST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check every manifest path actually exists
# ------------------------------------------------------------

path_columns = [
    "wav",
    "transcript",
    "egemaps",
    "mfcc"
]

for col in path_columns:
    m[f"{col}_exists"] = m[col].apply(
        lambda x: isinstance(x, str) and os.path.isfile(x)
    )

print("\nFILE EXISTENCE CHECK")
print("-" * 70)

for col in path_columns:
    count = m[f"{col}_exists"].sum()
    print(f"{col:12s}: {count}/{len(m)} exist")

# ------------------------------------------------------------
# 2. Show anything marked as a path but actually missing
# ------------------------------------------------------------

print("\nMISSING FILES REFERENCED BY MANIFEST")
print("-" * 70)

for col in path_columns:
    bad = m[
        m[col].notna() &
        ~m[f"{col}_exists"]
    ]

    if len(bad) > 0:
        print(f"\n{col.upper()} missing:")
        print(
            bad[["pid", "split", col]]
            .to_string(index=False)
        )
    else:
        print(f"{col.upper()}: none")

# ------------------------------------------------------------
# 3. Check participant folders
# ------------------------------------------------------------

print("\nPARTICIPANT FOLDER CHECK")
print("-" * 70)

folder_missing = []

for pid in m["pid"]:
    folder = os.path.join(ROOT, f"{int(pid)}_P")

    if not os.path.isdir(folder):
        folder_missing.append(pid)

print(
    "Missing participant folders:",
    len(folder_missing)
)

if folder_missing:
    print(folder_missing)

# ------------------------------------------------------------
# 4. Check actual participant folders that are NOT in manifest
# ------------------------------------------------------------

actual_folders = []

for name in os.listdir(ROOT):

    full_path = os.path.join(ROOT, name)

    if (
        os.path.isdir(full_path)
        and name.endswith("_P")
    ):
        try:
            pid = int(name.split("_")[0])
            actual_folders.append(pid)
        except ValueError:
            pass

actual_folders = sorted(actual_folders)

manifest_pids = set(m["pid"].astype(int))

not_in_manifest = [
    pid
    for pid in actual_folders
    if pid not in manifest_pids
]

print("\nACTUAL FOLDERS NOT IN MANIFEST")
print("-" * 70)

print(
    "Actual participant folders:",
    len(actual_folders)
)

print(
    "Not represented in manifest:",
    len(not_in_manifest)
)

if not_in_manifest:
    print(not_in_manifest)

# ------------------------------------------------------------
# 5. Final consistency check
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL CHECK")
print("=" * 70)

required = (
    m["wav_exists"] &
    m["transcript_exists"]
)

print(
    "Participants with valid audio + transcript:",
    required.sum()
)

print(
    "Participants without valid audio + transcript:",
    (~required).sum()
)

print("\nMissing participants:")
print(
    m.loc[
        ~required,
        ["pid", "split", "label", "phq"]
    ].to_string(index=False)
)

VERIFYING MANIFEST

FILE EXISTENCE CHECK
----------------------------------------------------------------------
wav         : 275/275 exist
transcript  : 275/275 exist
egemaps     : 275/275 exist
mfcc        : 275/275 exist

MISSING FILES REFERENCED BY MANIFEST
----------------------------------------------------------------------
WAV: none
TRANSCRIPT: none
EGEMAPS: none
MFCC: none

PARTICIPANT FOLDER CHECK
----------------------------------------------------------------------
Missing participant folders: 0

ACTUAL FOLDERS NOT IN MANIFEST
----------------------------------------------------------------------
Actual participant folders: 275
Not represented in manifest: 0

FINAL CHECK
Participants with valid audio + transcript: 275
Participants without valid audio + transcript: 0

Missing participants:
Empty DataFrame
Columns: [pid, split, label, phq]
Index: []
